SHAP可解释性图像

1.构建lgbm模型

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 

#plt全局设置
plt.rcParams['font.sans-serif'] = 'SimHei'  #字体
plt.rcParams['axes.unicode_minus'] = False  #负号正常显示

df = pd.read_excel('california.xlsx')

from sklearn.model_selection import train_test_split

X = df.drop(['price'],axis=1)
y = df['price']

#划分train-val-test=7：1：2
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.125, random_state=42)  # 0.125 x 0.8 = 0.1

# 数据集标准化
x_mean = X_train.mean()
x_std =  X_train.std()
y_mean = y.mean()
y_std = y.std()
X_train = (X_train - x_mean)/x_std
y_train = (y_train-y_mean)/y_std
X_val = (X_val - x_mean)/x_std
y_val = (y_val - y_mean)/y_std
X_test = (X_test - x_mean)/x_std
y_test = (y_test - y_mean)/y_std

import lightgbm as lgb

# LightGBM模型参数
params_lgb = {
    'learning_rate': 0.02,          # 学习率，控制每一步的步长，用于防止过拟合。典型值范围：0.01 - 0.1
    'boosting_type': 'gbdt',        # 提升方法，这里使用梯度提升树（Gradient Boosting Decision Tree，简称GBDT）
    'objective': 'mse',             # 损失函数
    'metric': 'rmse',               # 评估指标
    'num_leaves': 127,              # 每棵树的叶子节点数量，控制模型复杂度。较大值可以提高模型复杂度但可能导致过拟合
    'verbose': -1,                  # 控制 LightGBM 输出信息的详细程度，-1表示无输出，0表示最少输出，正数表示输出更多信息
    'seed': 42,                     # 随机种子，用于重现模型的结果
    'n_jobs': -1,                   # 并行运算的线程数量，-1表示使用所有可用的CPU核心
    'feature_fraction': 0.8,        # 每棵树随机选择的特征比例，用于增加模型的泛化能力
    'bagging_fraction': 0.9,        # 每次迭代时随机选择的样本比例，用于增加模型的泛化能力
    'bagging_freq': 4               # 每隔多少次迭代进行一次bagging操作，用于增加模型的泛化能力
}

model_lgb = lgb.LGBMRegressor(**params_lgb)
model_lgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], 
              eval_metric='rmse')  

2.摘要图（summary ploy）

In [ ]:
import shap
# 构建 shap解释器
explainer = shap.TreeExplainer(model_lgb)
# 计算测试集的shap值
shap_values = explainer.shap_values(X_train)
# 特征标签
labels = X_train.columns
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = 'Times new Roman'
plt.rcParams['font.size'] = 13
#cmap="?"配色viridis Spectral coolwar mRdYlGn RdYlBu RdBu RdGy PuOr BrBG PRGn PiYG
plt.figure()
shap.summary_plot(shap_values, X_train, feature_names=labels, plot_type="dot")

3.特征重要性排序

In [ ]:
# 计算train-val-test的SHAP值
shap_values_train = explainer.shap_values(X_train)
shap_values_val = explainer.shap_values(X_val)
shap_values_test = explainer.shap_values(X_test)

# 绘制SHAP值总结图（Summary Plot）
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
shap.summary_plot(shap_values_train, X_train, plot_type="bar", show=False)
plt.title("X_train")
plt.xlabel('')  # 移除 x 轴标签避免x轴重叠

plt.subplot(1, 3, 2)
shap.summary_plot(shap_values_val, X_val, plot_type="bar", show=False)
plt.title("X_val")

plt.subplot(1, 3, 3)
shap.summary_plot(shap_values_test, X_test, plot_type="bar", show=False)
plt.title("X_test")
plt.xlabel('')  

plt.tight_layout()
plt.show()

4.依赖图（特征间的关系）

In [ ]:
shap.dependence_plot('特征名1', shap_values, X_train, interaction_index='特征名2')

4.2交互作用摘要图（Interaction Summary Plot）（特征间的关系）

In [ ]:
shap_interaction_values = explainer.shap_interaction_values(X_train)
shap.summary_plot(shap_interaction_values, X_train)

5.力图（单个样本对结果的影响）

In [ ]:
# 绘制单个样本的SHAP解释（Force Plot）
sample_index = 7  # 选择一个样本索引进行解释
shap.force_plot(explainer.expected_value, shap_values_test[sample_index], X_test.iloc[sample_index], matplotlib=True)

6.SHAP热图（Heatmap）

In [ ]:
# 创建 shap.Explanation 对象
shap_explanation = shap.Explanation(values=shap_values_test[0:500,:], 
                                    base_values=explainer.expected_value, 
                                    data=X_test.iloc[0:500,:], feature_names=X_test.columns)
# 绘制热图
shap.plots.heatmap(shap_explanation)